# CATALYST RISK - Comprehensive Technical Guide

## Stochastic Catastrophe Risk Modeling Platform

This notebook provides a deep dive into the mathematical foundations, architectural components, and quantitative methodologies behind the CATALYST RISK platform. We'll explore:

1. **Monte Carlo Simulation Engine** - Vectorized stochastic modeling
2. **Hazard Generation** - Stochastic event intensity distributions
3. **Vulnerability Modeling** - Building-specific damage curves
4. **Financial Structures** - Deductibles, limits, and loss calculations
5. **Risk Metrics** - AAL, PML, and Exceedance Probability curves
6. **Sensitivity Analysis** - Parameter uncertainty impact
7. **Geographic Analysis** - Spatial risk distribution
8. **Model Validation** - Mathematical bounds and diagnostics

## 1. System Architecture Overview

```
┌─────────────────────────────────────────────────────────┐
│                    STREAMLIT UI                         │
│  (Multi-page Dashboard, Parameter Controls, Charts)    │
└────────────────────┬────────────────────────────────────┘
                     │
                     ▼
┌─────────────────────────────────────────────────────────┐
│              MONTE CARLO ENGINE                         │
│  (Vectorized NumPy Operations, Matrix Calculations)    │
└───┬───────────────┬───────────────┬────────────────────┘
    │               │               │
    ▼               ▼               ▼
┌─────────┐   ┌──────────┐   ┌─────────────┐
│ HAZARD  │   │VULNERAB- │   │   FINANCIAL  │
│ GENER-  │   │ILITY     │   │  STRUCTURES │
│ ATION   │   │CURVES    │   │             │
└─────────┘   └──────────┘   └─────────────┘
```

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import scipy.stats as stats

np.random.seed(42)
print("Libraries loaded successfully!")

## 2. Stochastic Hazard Event Generation

Catastrophe events are modeled using stochastic intensity distributions with log-normal distributions to capture long-tail disaster severities.

In [ ]:
def generate_hazard_intensity(num_sims, base_intensity, severity_multiplier):
    mu = np.log(base_intensity * severity_multiplier)
    sigma = 0.4
    intensities = np.random.lognormal(mu, sigma, num_sims)
    return intensities

num_sims = 10000
base_intensity = 100.0
severity_multiplier = 1.0

intensities = generate_hazard_intensity(num_sims, base_intensity, severity_multiplier)

print(f"Generated {num_sims:,} hazard events")
print(f"Mean intensity: {np.mean(intensities):.2f}")
print(f"95th percentile: {np.percentile(intensities, 95):.2f}")

### 2.1 Stochastic Hazard Density Distribution

In [ ]:
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Probability Density Function', 'Cumulative Distribution', 
                    'Q-Q Plot (Normality Check)', 'Box Plot with Statistics'),
    specs=[[{'type': 'scatter'}, {'type': 'scatter'}],
           [{'type': 'scatter'}, {'type': 'box'}]]
)

x_vals = np.linspace(intensities.min(), intensities.max(), 500)
kde = stats.gaussian_kde(intensities)
pdf_vals = kde(x_vals)

fig.add_trace(
    go.Scatter(
        x=x_vals, y=pdf_vals,
        mode='lines',
        name='PDF',
        line=dict(color='#3B82F6', width=3),
        fill='tozeroy',
        fillcolor='rgba(59, 130, 246, 0.3)'
    ),
    row=1, col=1
)

sorted_intensities = np.sort(intensities)
cdf = np.arange(1, len(sorted_intensities) + 1) / len(sorted_intensities)

fig.add_trace(
    go.Scatter(
        x=sorted_intensities, y=cdf,
        mode='lines',
        name='CDF',
        line=dict(color='#8B5CF6', width=3)
    ),
    row=1, col=2
)

theoretical_quantiles = stats.norm.ppf(cdf)
sample_quantiles = (sorted_intensities - np.mean(intensities)) / np.std(intensities)

fig.add_trace(
    go.Scatter(
        x=theoretical_quantiles, y=sample_quantiles,
        mode='markers',
        name='Q-Q',
        marker=dict(color='#EC4899', size=4, opacity=0.6)
    ),
    row=2, col=1
)

fig.add_trace(
    go.Scatter(
        x=[-3, 3], y=[-3, 3],
        mode='lines',
        name='Reference',
        line=dict(color='#6B7280', width=2, dash='dash')
    ),
    row=2, col=1
)

fig.add_trace(
    go.Box(
        y=intensities,
        name='Intensity',
        boxpoints='outliers',
        marker_color='#06B6D4',
        boxmean='sd'
    ),
    row=2, col=2
)

fig.update_layout(
    height=800,
    title_text='Stochastic Hazard Intensity Distribution Analysis',
    title_font_size=20,
    showlegend=False,
    template='plotly_dark'
)

fig.show()

## 3. Vulnerability Modeling

Logistic functions model the non-linear relationship between hazard intensity and damage ratio.

In [ ]:
def logistic_vulnerability(intensity, mid, k, cap):
    return cap / (1 + np.exp(-k * (intensity - mid)))

vulnerability_params = {
    'Concrete': {'mid': 140.0, 'k': 0.032, 'cap': 0.85, 'color': '#3B82F6'},
    'Wood': {'mid': 90.0, 'k': 0.045, 'cap': 0.97, 'color': '#10B981'},
    'Steel': {'mid': 155.0, 'k': 0.030, 'cap': 0.70, 'color': '#F59E0B'},
    'Masonry': {'mid': 105.0, 'k': 0.038, 'cap': 0.92, 'color': '#EF4444'}
}

intensity_range = np.linspace(0, 200, 500)
print("Vulnerability Parameters loaded for 4 building types")

### 3.1 2D Bounded Logistic Vulnerability Curves

In [ ]:
fig = go.Figure()

for b_type, params in vulnerability_params.items():
    damage_ratios = logistic_vulnerability(intensity_range, 
                                           params['mid'], 
                                           params['k'], 
                                           params['cap'])
    
    fig.add_trace(go.Scatter(
        x=intensity_range,
        y=damage_ratios,
        mode='lines',
        name=b_type,
        line=dict(color=params['color'], width=3)
    ))

fig.update_layout(
    title='Building-Specific Vulnerability Curves (Logistic Functions)',
    xaxis_title='Hazard Intensity',
    yaxis_title='Mean Damage Ratio (MDR)',
    yaxis_tickformat='.0%',
    template='plotly_dark',
    height=600
)

fig.show()

### 3.2 3D Bivariate Vulnerability Surface

In [ ]:
intensity_range = np.linspace(0, 200, 100)
quality_range = np.linspace(0.5, 1.5, 50)
intensity_grid, quality_grid = np.meshgrid(intensity_range, quality_range)

params = vulnerability_params['Wood']
adjusted_mid = params['mid'] * quality_grid
damage_surface = params['cap'] / (1 + np.exp(-params['k'] * (intensity_grid - adjusted_mid)))

fig = go.Figure(data=[go.Surface(
    x=intensity_range,
    y=quality_range,
    z=damage_surface,
    colorscale='Viridis',
    colorbar=dict(title='Damage Ratio'),
    contours=dict(z=dict(show=True, usecolormap=True, highlightcolor="limegreen", project_z=True))
)])

fig.update_layout(
    title='3D Vulnerability Surface: Wood Construction',
    scene=dict(
        xaxis_title='Hazard Intensity',
        yaxis_title='Construction Quality Multiplier',
        zaxis_title='Damage Ratio',
        zaxis_tickformat='.0%'
    ),
    height=700,
    template='plotly_dark'
)

fig.show()

## 4. Financial Structures

In [ ]:
def calculate_financials(gross_loss, deductible, limit):
    net_loss = np.maximum(gross_loss - deductible, 0)
    net_loss = np.minimum(net_loss, limit)
    return net_loss

tiv = 1_000_000
deductible = 50_000
limit = 750_000

damage_ratios = np.linspace(0, 1, 100)
gross_losses = tiv * damage_ratios
net_losses = calculate_financials(gross_losses, deductible, limit)

print(f"Property TIV: ${tiv:,}")
print(f"At 100% damage: Gross Loss: ${gross_losses[-1]:,.0f}, Net Loss: ${net_losses[-1]:,.0f}")

### 4.1 Financial Reconciliation Waterfall Chart

In [ ]:
fig = go.Figure(go.Waterfall(
    name="Financial Structure",
    orientation="v",
    measure=["relative", "relative", "relative", "total"],
    x=["Gross Loss", "Deductible", "Limit Cap", "Net Payout"],
    y=[gross_losses[-1], -deductible, -(gross_losses[-1] - deductible - limit), net_losses[-1]],
    text=[f"${gross_losses[-1]:,.0f}", f"-${deductible:,}", f"-${(gross_losses[-1] - deductible - limit):,.0f}", f"${net_losses[-1]:,.0f}"],
    textposition="outside",
    decreasing={"marker":{"color":"#EF4444"}},
    increasing={"marker":{"color":"#10B981"}},
    totals={"marker":{"color":"#3B82F6"}}
))

fig.update_layout(
    title="Financial Reconciliation: From Gross Damage to Net Payout",
    xaxis_title="Financial Component",
    yaxis_title="Loss Amount ($)",
    template="plotly_dark",
    height=500
)

fig.show()

## 5. Monte Carlo Simulation Engine

In [ ]:
def run_simulation(num_sims, num_properties, tivs, building_types, hazard_intensity, vulnerability_params):
    local_variation = np.random.normal(1.0, 0.1, num_properties)
    local_variation = np.clip(local_variation, 0.7, 1.3)
    intensity_matrix = np.outer(hazard_intensity, local_variation)
    
    damage_ratio_matrix = np.zeros_like(intensity_matrix)
    
    for b_type in np.unique(building_types):
        mask = (building_types == b_type)
        if mask.any():
            params = vulnerability_params[b_type]
            dr = logistic_vulnerability(intensity_matrix[:, mask], params['mid'], params['k'], params['cap'])
            damage_ratio_matrix[:, mask] = dr
    
    gul_matrix = tivs * damage_ratio_matrix
    net_matrix = calculate_financials(gul_matrix, 25000, 750000)
    portfolio_losses = np.sum(net_matrix, axis=1)
    
    return portfolio_losses, gul_matrix, net_matrix

num_properties = 500
num_sims = 10000

tivs = np.random.lognormal(np.log(500000), 0.5, num_properties)
building_types = np.random.choice(list(vulnerability_params.keys()), num_properties, p=[0.3, 0.25, 0.25, 0.2])
hazard_intensities = generate_hazard_intensity(num_sims, 100.0, 1.0)

portfolio_losses, gul_matrix, net_matrix = run_simulation(num_sims, num_properties, tivs, building_types, hazard_intensities, vulnerability_params)

print(f"Simulation Complete! Properties: {num_properties:,}, Simulations: {num_sims:,}")
print(f"AAL: ${np.mean(portfolio_losses):,.0f}")

### 5.1 Probabilistic Loss Variance (Violin Plots)

In [ ]:
building_type_losses = {}
for b_type in np.unique(building_types):
    mask = (building_types == b_type)
    building_type_losses[b_type] = net_matrix[:, mask].mean(axis=1)

fig = go.Figure()
for i, (b_type, losses) in enumerate(building_type_losses.items()):
    fig.add_trace(go.Violin(
        y=losses,
        name=b_type,
        box_visible=True,
        meanline_visible=True,
        fillcolor=vulnerability_params[b_type]['color'],
        opacity=0.7
    ))

fig.update_layout(
    title='Probabilistic Loss Distribution by Building Type',
    yaxis_title='Mean Loss per Property ($)',
    template='plotly_dark',
    height=600
)

fig.show()

## 6. Risk Metrics Calculation

In [ ]:
def calculate_aal(losses):
    return np.mean(losses)

def calculate_pml(sorted_losses, return_period):
    num_sims = len(sorted_losses)
    index = int(np.floor(num_sims / return_period)) - 1
    index = max(0, min(index, num_sims - 1))
    return sorted_losses[index]

def generate_ep_curve(losses):
    sorted_losses = np.sort(losses)[::-1]
    return_periods = [1, 2, 5, 10, 25, 50, 100, 250]
    ep_data = []
    for rp in return_periods:
        loss_val = calculate_pml(sorted_losses, rp)
        ep_data.append({'return_period': rp, 'probability': 1.0 / rp, 'loss': loss_val})
    return pd.DataFrame(ep_data)

sorted_losses = np.sort(portfolio_losses)[::-1]
aal = calculate_aal(portfolio_losses)
pml_100 = calculate_pml(sorted_losses, 100)
pml_250 = calculate_pml(sorted_losses, 250)
ep_df = generate_ep_curve(portfolio_losses)

print(f"AAL: ${aal:,.0f}")
print(f"100-Year PML: ${pml_100:,.0f}")
print(f"250-Year PML: ${pml_250:,.0f}")

### 6.1 The OEP (Occurrence Exceedance Probability) Curve

In [ ]:
sorted_losses = np.sort(portfolio_losses)[::-1]
sorted_losses = sorted_losses[sorted_losses > 0]
num_events = len(sorted_losses)
total_sims = len(portfolio_losses)

indices = np.linspace(0, num_events - 1, min(200, num_events), dtype=int)
plot_losses = sorted_losses[indices]
ranks = indices + 1
probabilities = ranks / total_sims
return_periods = 1.0 / probabilities

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=return_periods,
    y=plot_losses,
    mode='lines',
    name='EP Curve',
    line=dict(color='#3B82F6', width=3)
))

fig.add_trace(go.Scatter(
    x=ep_df['return_period'],
    y=ep_df['loss'],
    mode='markers',
    name='Standard RP',
    marker=dict(size=10, color='#EF4444')
))

fig.update_layout(
    title='Exceedance Probability (EP) Curve',
    xaxis_title='Return Period (Years)',
    yaxis_title='Loss Amount ($)',
    xaxis_type='log',
    template='plotly_dark',
    height=600
)

fig.show()

## 7. Sensitivity Analysis

In [ ]:
sensitivity_scenarios = {
    'Climate Change (+15%)': 1.15,
    'Climate Change (-15%)': 0.85,
    'Inflation (+10%)': 1.10,
    'Improved Building Codes (-20%)': 0.80
}

results = []
base_aal = calculate_aal(portfolio_losses)

for scenario, shift in sensitivity_scenarios.items():
    shifted_intensities = hazard_intensities * shift
    shifted_losses, _, _ = run_simulation(num_sims, num_properties, tivs, building_types, shifted_intensities, vulnerability_params)
    shifted_aal = calculate_aal(shifted_losses)
    aal_change = ((shifted_aal - base_aal) / base_aal) * 100
    results.append({'Scenario': scenario, 'Change (%)': aal_change})

sensitivity_df = pd.DataFrame(results)
print(sensitivity_df.to_string(index=False))

### 7.1 Model Sensitivity (Tornado Chart)

In [ ]:
fig = go.Figure(go.Bar(
    x=sensitivity_df['Change (%)'],
    y=sensitivity_df['Scenario'],
    orientation='h',
    marker_color=sensitivity_df['Change (%)'].apply(lambda x: '#EF4444' if x > 0 else '#10B981'),
    text=sensitivity_df['Change (%)'].apply(lambda x: f"{x:+.1f}%"),
    textposition='outside'
))

fig.update_layout(
    title='Model Sensitivity: AAL Impact of Parameter Shifts',
    xaxis_title='AAL Change (%)',
    template='plotly_dark',
    height=500
)

fig.show()

## 8. Geographic Risk Analysis

In [ ]:
center_lat, center_lon = 34.0522, -118.2437
lats = np.random.normal(center_lat, 0.125, num_properties)
lons = np.random.normal(center_lon, 0.125, num_properties)
property_mean_losses = net_matrix.mean(axis=0)

geo_df = pd.DataFrame({
    'lat': lats, 'lon': lons, 'tiv': tivs,
    'building_type': building_types,
    'expected_loss': property_mean_losses,
    'loss_ratio': property_mean_losses / tivs
})

print(f"Geographic analysis complete: {len(geo_df)} properties mapped")

### 8.1 Geospatial Risk Concentration (Heatmap Density)

In [ ]:
fig = go.Figure(go.Scatter(
    x=geo_df['lon'],
    y=geo_df['lat'],
    mode='markers',
    marker=dict(
        size=geo_df['tiv'] / 50000,
        color=geo_df['loss_ratio'],
        colorscale='Viridis',
        colorbar=dict(title='Loss Ratio'),
        opacity=0.7
    )
))

fig.update_layout(
    title='Geographic Risk Concentration: Portfolio Spatial Distribution',
    xaxis_title='Longitude',
    yaxis_title='Latitude',
    template='plotly_dark',
    height=600
)

fig.show()

## 9. Model Validation

In [ ]:
def run_validation_checks(gul_matrix, net_matrix):
    checks = []
    
    max_net = net_matrix.max()
    checks.append({'metric': 'Net Loss Finite', 'passed': np.isfinite(max_net), 'detail': f'Max net loss: ${max_net:,.0f}'})
    
    min_net = net_matrix.min()
    checks.append({'metric': 'Net Loss >= 0', 'passed': min_net >= -1e-6, 'detail': f'Min net loss: ${min_net:,.0f}'})
    
    check3 = np.all(gul_matrix >= net_matrix - 1e-6)
    checks.append({'metric': 'GUL >= Net Loss', 'passed': check3, 'detail': 'All GUL >= Net Loss'})
    
    return checks

validation_results = run_validation_checks(gul_matrix, net_matrix)
for check in validation_results:
    status = "PASS" if check['passed'] else "FAIL"
    print(f"{status} | {check['metric']}: {check['detail']}")

## 10. Summary

This notebook demonstrates the CATALYST RISK platform's mathematical foundations:

- **Vectorized Monte Carlo**: Sub-second 10,000+ simulations
- **Stochastic Hazard Modeling**: Log-normal distributions for realistic disaster severities
- **Vulnerability Curves**: Building-specific logistic functions
- **Financial Structures**: Proper deductible/limit implementation
- **Risk Metrics**: AAL, PML, EP curves following industry standards
- **Sensitivity Analysis**: Parameter uncertainty impact
- **Geographic Analysis**: Spatial risk distribution
- **Model Validation**: Comprehensive diagnostic checks

This platform demonstrates enterprise-grade quantitative risk modeling capabilities suitable for insurance and reinsurance applications.